<a href="https://colab.research.google.com/github/Le2se0hy/FA_ProAn/blob/main/%ED%95%B8%EC%A6%88%EC%98%A8%EB%A8%B8%EC%8B%A0%EB%9F%AC%EB%8B%9D_3%EC%9E%A5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 핸즈온 머신러닝_3장 [분류]

0210(화) - p.143 ~ 146

0211(수) - p.147

0212(목) - p.147 ~ 149

0213(금) - p.149 ~ 150

0216(월) - p.151 ~ 153

0218(수) - p.153 ~ 154

0219(목) - p.154 ~ 156

0221(토) - p.156 ~ 158 (금요일에 잠들어버려서 메꾸기)

...

0308(일) = ~ p.172

---

## 3.1 MNIST

- MNIST 데이터셋 : 손으로 쓴 70,000개의 작은 숫자 이미지. 각 이미지에는 어떤 숫자를 나타내는지 레이블되어 있고 학습용으로 아주 많이 사용됨



In [ ]:
# OpenML.org에서 MNIST를 내려받는 코드

from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', as_frame=False)

sklearn.datasets 패키지에 있는   세 종류의 함수
- fetch_* : fetch_openml()과 같이 실전 데이터셋을 다운로드하는 함수
- load_* : 사이킷런에 번들로 포함된 소규모 데이터셋을 로드하기 위한 함수 (인터넷에서 다운로드 필요x)
- make_* : 테스트에서 유용한 가짜 데이터셋을 생성하기 위한 함수. 생성된 데이터셋을 보통 넘파이 배열이고 (X,y) 튜플로 반환됨

sklearn.utils.Bunch 객체로 반환되는 데이터셋도 있는데 이 객체는 다음과 같은 항목을 참조할 수 잇는 딕셔너리
- DESCR : 데이터셋 설명
- data : 입력 데이터. 일반적으로 2D 넘파이 배열
- target : 레이블. 일반적으로 1D 넘파이 배열

fetch_openml() 함수는 기본적으로 입력을 판다스 데이터프레임, 레이블을 판다스 시리즈로 반환함.

하지만 MNIST 데이터셋은 이미지이므로 데이터프레임이 잘 맞지 않음. → as_frame=False로 지정하여 넘파이 배열로 데이터를 받음

In [ ]:
X, y = mnist.data, mnist.target
X
# array([[0, 0, 0, ..., 0, 0, 0],
#       [0, 0, 0, ..., 0, 0, 0],
#       [0, 0, 0, ..., 0, 0, 0],
#       ...,
#       [0, 0, 0, ..., 0, 0, 0],
#       [0, 0, 0, ..., 0, 0, 0],
#       [0, 0, 0, ..., 0, 0, 0]])

X.shape
# (70000, 784)

y
# array(['5', '0', '4', ..., '4', '5', '6'], dtype=object)

y.shape
#
# (70000,)

이미지가 70,000개 있고 각 이미지에 784개의 특성이 있음 = 이미지가 28x28 픽셀이기 때문

각각의 특성은 단순히 0(흰색) ~ 255(검은색)까지의 픽셀 강도를 나타냄

데이터셋에서 이미지 하나를 확인해보자.

샘플의 특성 베터를 추출해서 28x28 배열로 크기를 바꾸고 맷플롯립의 imshow() 함수를 사용해 그리면 됨. cmap="binary"로 지정해 grayscale 맵을 사용

In [ ]:
import matplotlib.pyplot as plt

def plot_digit(image_data):
  image = image_data.reshape(28, 28)  # image_data는 보통 길이 784(=28×28)인 1차원 배열(벡터).
                                      # reshape(28, 28)은 그 벡터를 2차원 이미지 형태(28행 28열)로 바꾸는 것.

  plt.imshow(image, cmap="binary")
  plt.axis("off")

some_digit = X[0]
plot_digit(some_digit)
plt.show()
y[0]

데이터를 자세히 조사하기 전에 항상 테스트 세트를 만들고 따로 떼어놓아야 함.

fetch_openml()이 반환한 MNIST 데이터셋은 이미 훈련 세트(앞쪽 60,000개)와 테스트 세트(뒤쪽 10,000개)로 나뉘어있음

In [ ]:
X_train, X_test, y_train, y_test = X[:60000], X[60000:], y[:60000], y[60000:]

---
## 3.2 이진 분류기 훈련

5-감지기는 '5'와 '5아님' 두 개의 클래스를 구분할 수 있는 이진 분류기

In [ ]:
y_train_5 = (y_train =='5') # 5는 True고. 다른 숫자는 모두 False
y_test_5 = (y_test =='5')

분류 모델을 하나 선택해서 훈련해보자.

- **확률적 경사 하강법 SDG**: 사이킷런의 SGDClassifier 클래스를 사용하며 매우 큰 데이터셋을 효율적으로 처리할 수 있다는 장점이 있음. 한번에 하나씩 훈련 샘플을 독립적으로 처리할 수 있음

In [ ]:
from sklearn.linear_model import SGDClassifier

sgd_clf = SGDClassifier(random_state=42)
sgd_clf.fit(X_train, y_train_5)

# 이 모델을 사용해 숫자 5의 이미지를 감지해보자면
sgd_clf.predict([some_digit])

# 값이 True가 나왔다면 이미지가 5를 나타낸다고 추측함.
# 해당 샘플에 대해서는 정확히 맞춤 !!

---
## 3.3 성능 측정

### 3.3.1 교차 검증을 사용한 정확도 측정

cross_val_score() 함수로 폴드가 3개인 k-폴드 교차 검증을 사용해 SGDClassifier 모델을 평가해보자.



In [ ]:
from sklearn.model_selection import cross_val_score
cross_val_score(sgd_clf, X_train, y_train_5, cv=3, scoring="accuracy")

모든 교차 검증 폴드에 대해 정확도가 95% 이상임 !!

모든 이미지를 가장 많이 등장하는 클래스로 분류하는 더미 분류기를 만들어 비교함

In [ ]:
from sklearn.dummy import DummyClassifier

dummy_clf = DummyClassifier()
dummy_clf.fit(X_train, y_train_5)
# False가 출력됩니다. 즉, True로 예측된 것이 없습니다.
print(any(dummy_clf.predict(X_train)))

cross_val_score(dummy_clf, X_train, y_train_5, cv=3, scoring="accuracy")

정확도가 90% 이상으로 나옴! 이미지의 10% 정도만 숫자 5이기 때문에 무조건 '5아님'으로 예측하면 정확히 맞출 확률이 90%이다.

이 예제는 정확도를 분류기의 성능 측정 지표로 선호하지 않는 이유를 보여줌. 특히 불균형한 데이터셋을 다룰 때 더욱 그럼.

분류기의 성능을 평가하는 더 좋은 방법은 오차 행렬을 조사하는 것이다.

### 3.3.2 오차 행렬

오차 행렬의 기본 아이디어는 모든 A/B 쌍에 대해 클래스 A의 샘플이 클래스 B로 분류된 횟수를 세는 것이다.

예 ) 분류기가 숫자 8의 이미지를 0으로 잘못 분류한 횟수를 알고 싶다면 오차 행렬에서 8번 행의 0번 열을 보면 됨

오차 행렬을 만들려면 실제 타깃과 비교할 수 있도록 예측값을 만들어야 함. (테스트 세트로 만들 순 있지만 여기서 사용X)

→ cross_val_predict() 함수 사용

In [ ]:
from sklearn.model_selection import cross_val_predict

y_train_pred = cross_val_predict(sgd_clf, X_train, y_train_5, cv=3)

cross_val_predict() 함수는 k-폴드 교차 검증을 수행하지만 평가 점수를 반환하지 않고, 각 테스트 폴드에서 얻은 예측을 반환

 = 훈련 세트의 모든 샘플에 대해 깨끗한 예측을 얻게 됨 (깨끗==훈련하는 동안 보지 못했던 데이터에 대해 예측)


In [ ]:
# confusion_matrix() 함수를 사용해 오차행렬을 만들자
# 타깃 클래스 (y_train_5)와 예측 클래스 (y_train_pred)를 넣고 호출하면 됨

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_train_5, y_train_pred)
cm

오차 행렬의 행 = 실제 클래스, 열 = 예측한 클래스를 나타냄

이 행렬의 첫 번째 행은 '5 아님' 이미지(음성 클래스)에 대한 것으로 53892개를 '5 아님'으로 정확하게 분류했고(진짜 음성) 나머지 687개는 '5'라고 잘못 분류함 (거짓 양성)

두 번째 행은 '5' 이미지(양성 클래스)에 대한 것으로 1,891개를 '5 아님'으로 잘못 분류했고(거짓 음성 또는 2종오류) ㄴ아머지 3,530개를 정확히 '5'라고 분류함(진짜양성)

완벽한 분류기라면 진짜 양성과 진짜 음성만 가지고 있을 것이므로 오차행렬의 주대각성만 0이 아닌 값이 됨

In [ ]:
y_train_perfect_predictions = y_train_5
confusion_matrix(y_train_5, y_train_perfect_predictions)

**정밀도** : 오차행렬보다 조금 더 요약된 지표로 양성 예측의 정확도를 뜻한다.


$ 정밀도  = \frac{TP}{TP + FP} $

$TP$ = 진짜 양성의 수, $FP$ = 거짓 양성의 수

가장 간단한 방법은 제일 확신이 높은 샘플에 대해 양성 예측을 하고 나머지는 모두 음성 예측을 하는 분류기 →  양성 예측이 맞는다면 정밀도는 100%

하지만 이런 분류기는 다른 모든 양성 샘플을 무시하기에 유용하지 않음

**재현율**(=민감도,TPR) : 정밀도와 같이 사용하는 지표. 분류기가 정확하게 감지한 양성 샘플의 비율을 뜻함

$재현율 = \frac{TP}{TP + FN}$

$FN$ = 거짓 음성의 수






### 3.3.3 정밀도와 재현율

사이킷런은 정밀도와 재현율을 포함하여 분류기의 지표를 계산하는 여러 함수를 제공

In [ ]:
from sklearn.metrics import precision_score, recall_score
precision_score(y_train_5, y_train_pred) # == 3530 / (687 + 3530)


In [ ]:
recall_score(y_train_5, y_train_pred) # == 3530 / (1891 + 3530)

'5-감지기'가 정확도에서 봤을 떄만큼 좋진 않음

5로 판별된 이미지 중 83.7%만 정확하고 전체 숫자 5에서 65.1%만 감지함

**$F_1$점수** : 정밀도와 재현율의 조화평균(보통의 평균보다 낮은 값에 비중을 더 둠)

$ F_1 score = \frac{2}{\frac{1}{정밀도} + \frac{1}{재현율}} = 2 \times \frac{정밀도 × 재현율}{정밀도 + 재현율} = \frac{TP}{TP + \frac{FN + FP}{2}}$

f1_score() 함수를 호출해서 계산

In [ ]:
from sklearn.metrics import f1_score
f1_score(y_train_5, y_train_pred)

정밀도와 재현율이 비슷한 분류기에서는 F_1 점수가 높음. 하지만 항상 바람직한건 X

정밀도가 더 높아야 할 때가 있고 재현율이 더 높아야 할 때가 있음 → 정밀도와 재현율 모두를 얻을순 x

= **정밀도/재현율 트레이드오프**

### 3.3.4 정밀도/재현율 트레이드오프

SSGClassifier가 분류를 어떻게 결정하는지 살펴보며 이해해보자

이 분류기는 결정함수를 사용하여 각 샘플의 점수를 계산 : 점수가 임곗값보다 크면 양성 클래스에 할당, 그렇지않으면 음성클래스에 할당



[이부분에 결정 임곗값 사진 넣고 작동]

사이킷런에서 임곗값을 직접 지정할 수는 없지만 예측에 사용한 점수는 확인할 수 있음

분류기의 predict() 메서드 대신 decision_function() 메서드를 호출하면 각 샘플의 점수를 얻을 수 있음

In [ ]:
y_scores = sgd_clf.decision_function([some_digit])
y_scores

In [ ]:
threshold = 0 # 임곗값
y_some_digit_pred = (y_scores > threshold)
y_some_digit_pred

여기서 SGDClassifier의 임곗값이 0이므로 predict() 메서드와 같은 결과(= True)를 반환함

임곗값을 높이면

In [ ]:
threshold = 3000
y_some_digit_pred = (y_scores > threshold)
y_some_digit_pred

이 결과는 임곗값을 높이면 재현율이 줄어든다는 것을 보여줌

이미지가 실제로 숫자 5이고 임곗값이 0일 때는 분류기가 이를 감지, 임곗값을 3000으로 높이면 이를 놓치게 됨

그렇다면 적절한 임곗값을 어떻게 정함?

→ 먼저 cross_val_predict() 함수를 사용해 훈련 세트에 있는 모든 샘플의 점수를 구해야 한다.

하지만 이번에는 예측 결과가 아니라 결정 점수를 반환하도록 지정해야함

In [ ]:
y_scores = cross_val_predict(sgd_clf, X_train, y_train_5, cv=3,
                             method="decision_function")

이 점수는 precision_recall_curve() 함수를 사용하여 가능한 모든 임곗값에 대해 정밀도와 재현율을 계산할 수 있음

(이 함수는 무한한 임곗값에 해당하는 값으로 마지막 정밀도에 1을, 마지막 재현율에 0을 추가함)


In [ ]:
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_train_5, y_scores)

plt.plot(thresholds, precisions[:-1], "b--", label="정밀도", linewidth=2)
plt.plot(thresholds, recalls[:-1], "g-", label="재현율", linewidth=2)
plt.vlines(threshold, 0, 1.0, "k", "dotted", label="임곗값")

idx = (abs(thresholds - threshold)).argmin()
plt.plot(thresholds[idx], precisions[idx], "bo", markersize=8)  # 정밀도 동그라미
plt.plot(thresholds[idx], recalls[idx], "go", markersize=8)      # 재현율 동그라미

# 축/레이블/범위
plt.xlabel("임곗값(Threshold)")
plt.ylabel("점수")
plt.title("Precision / Recall vs Threshold")
plt.xlim(thresholds.min(), thresholds.max())
plt.ylim(0, 1.05)

# 그리드 & 범례
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="best")

plt.show()


이 임곗값에서 정밀도는 약 90%이고 재현율은 약 50%이다.

좋은 정밀도/재현율 트레이드오프를 선택하는 다른 방법은 재현율에 대한 정밀도 곡선을 그리는 것이다.

In [ ]:
# plt.plot(recalls, precisions, linewidth=2, label="정밀도/재현율 곡선")
# 그리드, 범례, 레이블, 화살표, 텍스트를 추가합니다.
# plt.show()

# 해당 코드를 돌리기 위한 그리드, 범례, 레이블 등을 알지 못하겠어서 주석처리

재현율 80% 근처에선 정밀도가 급격하게 줄어들기 시작함. 이 하강점 직전을 정밀도/재현율 트레이드오프로 선택하는 것이 좋음

정밀도 90%를 달성하는 것이 목표라고 가정해보자. 그래프에서 사용할 임곗값을 찾을 수 있지만 정확하지 않음.

다른 방법은 정밀도가 최소 90%가 되는 가장 낮은 임곗값을 찾는거임.

넘파이 배열의 argmax() 메서드를 사용할 수 있음. 이 메서드는 최댓값의 첫 번째 인덱스를 반환함. 여기에선 첫 번째 True 값을 의미함

In [ ]:
idx_for_90_precision = (precisions >= 0.90).argmax()
threshold_for_90_precision = thresholds[idx_for_90_precision]
threshold_for_90_precision

훈련세트에 대한 예측을 만들려면 분류기의 predict() 메서드를 호출하는 대신 다음 코드를 실행

In [ ]:
y_train_pred_90 = (y_scores >= threshold_for_90_precision)

# 이 예측에 대한 정밀도와 재현율을 확인해봅시다

In [ ]:
precision_score(y_train_5, y_train_pred_90)

In [ ]:
recall_at_90_precision = recall_score(y_train_5, y_train_pred_90)
recall_at_90_precision

정밀도 90%를 달성한 분류기를 만들었음 !!

임곗값을 충분히 크게 지정하기만 하면 거의 모든 정밀도의 분류기를 손쉽게 만들 수 있음

하지만 재현율이 너무 낮다면 높은 정밀도의 분류기는 유용하지 X

많은 애플리케이션에 재현율 48%는 훌륭한 값이 아님

### 3.3.5 ROC 곡선

수신기 조작 특성(ROC) 곡선도 이진 분류에서 널리 사용되는 도구이다.

**ROC 곡선** : 거짓 양성 비율(FPR)에 대한 진짜 양성 비율(TPR)의 곡선

- 거짓 양성 비율 FPR : 양성으로 잘못 분류된 음성 샘플의 비율 = 1에서 TNR을 뺀 값
- 진짜 음성 비율 TNR : 음성으로 정확하게 분류한 음성 샘플의 비율 (= 특이도)

= 민감도에 대한 1-특이도 그래프

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_train_5, y_scores)

90% 정밀도에 해당하는 지점을 찾기 위해 원하는 임곗값의 인덱스를 찾아야 함. 임곗값이 내림차순으로 정렬되어 있기 때문에 첫 번째 라인에 >=가 아니라 <=를 사용

In [ ]:
idx_for_threshold_at_90 = (thresholds <= threshold_for_90_precision).argmax()
tpr_90, fpr_90 = tpr[idx_for_threshold_at_90], fpr[idx_for_threshold_at_90]

plt.plot(fpr, tpr, linewidth=2, label="ROC 곡선")
plt.plot([0, 1], [0, 1], 'k:', label="랜덤 분류기의 ROC 곡선")
plt.plot([fpr_90], [tpr_90], "ko", label="90% 정밀도에 대한 임곗값")
# 레이블, 그리드, 범례, 화살표, 텍스트를 추가합니다

# ROC 곡선 그리기
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, linewidth=2, label="ROC 곡선")
plt.plot([0, 1], [0, 1], 'k:', label="랜덤 분류기의 ROC 곡선")
plt.plot([fpr_90], [tpr_90], "ko", label="90% 정밀도에 대한 임곗값")

# 레이블 추가
plt.xlabel("거짓 양성 비율 (False Positive Rate)")
plt.ylabel("진짜 양성 비율 (True Positive Rate)")
plt.title("ROC 곡선")

# 그리드 추가
plt.grid(True)

# 범례 추가
plt.legend(loc="lower right")

# 화살표와 텍스트 추가
plt.annotate(
    "90% 정밀도 임곗값",
    xy=(fpr_90, tpr_90),                 # 점의 위치
    xytext=(fpr_90 + 0.15, tpr_90 - 0.1), # 텍스트 위치
    arrowprops=dict(facecolor="black", shrink=0.05, width=1, headwidth=8),
    fontsize=10
)

plt.show()

여기에도 트레이드오프가 있음. 재현율(TPR)이 높을수록 분류기가 만드는 거짓 양성 비율이 늘어남

점선은 랜덤 분류기의 ROC 곡선을 뜻한다. 좋은 분류기는 이 점선에서 최대한 멀리 떨어져 있어야 함

곡선 아래의 면적(AUC)을 측정해 분류기들을 비교할 수 있음

**완벽한 분류기는 ROC의 AUC가 1이고, 완전한 랜덤 분류기는 0.5임**



In [ ]:
from sklearn.metrics import roc_auc_score
roc_auc_score(y_train_5, y_scores)

RandomForestClassifier를 만들어 SGDClassifier의 PR곡선과 F_1점수를 비교해보자

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest_clf = RandomForestClassifier(random_state=42)

precision_recall_curve() 함수는 각 샘플에 대한 레이블과 점수를 기대함

따라서 랜덤 포레스트 분류기를 훈련하여 각 샘플에 점수를 부여해야 함. RandomForestClassifier는 작동 방식 때문에 decision_function()을 제공하지 않음

다행히 각 샘플에 대한 클래스 확률을 반환하는 predict_proba() 메서드를 제공

이 중 양성 클래스에 대한 확률을 점수로 사용할 수 있음. cross_val_predict() 함수를 호출하여 교차 검증으로 RandomForestClassifier를 훈련하고 모든 이미지에 대한 클래스 확률을 예측할 수 있음

In [ ]:
y_probas_forest = cross_val_predict(forest_clf, X_train, y_train_5, cv=3,
                                    method="predict_proba")

훈련 세트에 있는 처음 두 개의 이미지에 대한 클래스 확률을 확인해보자면

In [ ]:
y_probas_forest[:2]

이 모델은 첫 번째 이미지를 89%의 확률로 양성이라고 예측함

그리고 두 번째 이미지를 99% 확률로 음성이라 예측함.

모든 이미지는 양성 또는 음성 둘 중 하나이기 때문에 각 행의 확률을 더하면 100%가 됨


두 번째 열에 양성 클래스에 대한 추정 확률이 포함되어 있으므로 이를 precision_recall_curve() 함수에 전달함

In [ ]:
y_scores_forest = y_probas_forest[:, 1]
precisions_forest, recalls_forest, thresholds_forest = precision_recall_curve(
    y_train_5, y_scores_forest)

plt.plot(recalls_forest, precisions_forest, "b--", label="랜덤 포레스트")
plt.plot(recalls, precisions, "--", label="SGD")
# 레이블, 그리드, 범례를 추가

# 레이블 추가
plt.xlabel("재현율 (Recall)")
plt.ylabel("정밀도 (Precision)")
plt.title("정밀도-재현율 곡선 비교")

# 그리드 추가
plt.grid(True)

# 범례 추가
plt.legend(loc="lower left")

plt.show()

---
## 3.4 다중 분류

이중 분류기는 두 개의 클래스를 구별하는 반면 다중 분류기는 둘 이상의 클래스를 구별할 수 있음

이진 분류기를 여러 개 사용해 다중 클래스를 분류하는 기법도 많음

예를 들어 특정 숫자 하나만 구분하는 숫자별 이진 분류기 10개를 훈련시켜 클래스가 10개인 숫자 이미지 분류 시스템을 만들 수 있음

- OvR or OvA : 이미지를 분류할 때 각 분류기의 결정 점수 중에서 가장 높은 것을 클래스로 선택하면 됨

- OvO : 0과 1 구별, 0과 2 구별, 1과 2 구별 등과 같이 각 숫자의 조합마다 이진 분류기를 훈련시키는 것. 클래스가 N개라면 분류기는 N x (N-1)/2개가 필요.

일부 알고리즘은 훈련 세트의크기에 민감해서 큰 훈련 세트에서 몇 개의 분류기를 훈련시키는 쪽이 빠르므로 OvO를 선호, 하지만 대부분의 이진 분류 알고리즘에서는 OvR을 선호

다중 클래스 분류 작업에 이진 분류 알고리즘을 선택하면 사이킷런이 알고리즘에 따라 자동으로 OvR 또는 OvO를 실행 → 서포트 벡터 머신 분류기를 테스트 해보자 (200개의 이미지만 사용)







In [ ]:
from sklearn.svm import SVC

svm_clf = SVC(random_state=42)
svm_clf.fit(X_train[:2000], y_train[:2000]) # y_train_5가 아닌 y_train 사용해야함

# 한 이미지에 대한 예측 만들기
svm_clf.predict([some_digit])

이 코드는 실제로 클래스 쌍마다 하나씩 45번의 예측을 수행하여 가장 많은 쌍에서 승리한 클래스를 선택했음

decision_function() 메서드를 호출하면 샘플마다 총 10개의 점수를 반환하는 것을 볼 수 있음

각 클래스는 동률 문제를 해결하기 위해 분류기 점수를 기반으록 각 쌍에서 이긴 횟수에 약간의 조정 값을 더하거나 뺀 점수를 얻음

In [ ]:
some_digit_scores = svm_clf.decision_function([some_digit])
some_digit_scores.round(2)

가장 높은 점수는 9.3 → 클래스 5에 해당

In [ ]:
class_id = some_digit_scores.argmax()
class_id

분류가거 훈련될 때 classes_ 속서에 타깃 클래스의 리스트를 값으로 정렬하여 저장함

MNIST의 경우 classes_ 배열에 있는 각 클래스의 인덱스가 클래스의 값 자체와 같음 = 즉 인덱스 5에 해당하는 클래스의 값은 '5'

하지만 일반적으로 이런 경우는 드물어서 클래스 레이블을 확인해봐야함



In [ ]:
svm_clf.classes_

In [ ]:
svm_clf.classes_[class_id]

사이킷런에서 OvO나 OvR를 사용하도록 강제하려면 OneVsOneClassifier나 OneVsRestClassifier를 사용

간단하게 이진 분류기 인스턴스를 만들어 객체를 생성할 때 전달하면 됨(심지어 이진 분류기일 필요도 없음)

예를 들어 다음 코드는 SVC 기반으로 OvR 전략을 사용하는 다중 분류기를 만듦

In [ ]:
from sklearn.multiclass import OneVsRestClassifier

ovr_clf = OneVsRestClassifier(SVC(random_state=42))
ovr_clf.fit(X_train[:2000], y_train[:2000])

예측을 만들고 훈련된 분류기 개수를 확인해보면

In [ ]:
ovr_clf.predict([some_digit])

In [ ]:
len(ovr_clf.estimators_)

다중 분류 데이터셋에서 SGDClassifier를 훈련하고 예측을 만드는 것도 간단하다

In [ ]:
sgd_clf = SGDClassifier(random_state=42)
sgd_clf.fit(X_train, y_train)
sgd_clf.predict([some_digit])


예측이 틀림!..예측 오류가 날 수 있다

이번에는 사이킷런이 OvR 전략을 사용함. 10개의 클래스가 있기 때문에 10개의 이진 분류기를 훈련함 decision_function() 메서드는 클래스마다 하나의 값을 반환함

SGD 분류기가 각 클래스에 부여한 점수를 확인하자



In [ ]:
sgd_clf.decision_function([some_digit]).round()

이 결과에서 분류기가 예측 결과에 강한 확신을 보이고 있음을 알 수 있음

대부분의 점수가 큰 음수지만 클래스 3의 점수는 1,824이고 클래스 5도 -1,386으로 그리 멀리 떨어져 있지 않음.

각 클래스마다 거의 같은 개수의 이미지가 있기 때문에 정확도 지표가 좋음 → cross_val_score() 함수를 사용해 이 모델을 평가해보자

In [ ]:
cross_val_score(sgd_clf, X_train, y_train, cv=3, scoring="accuracy")

모든 테스트 폴드에서 85.8% 이상을 얻음!!

랜덤 분류기를 사용했다면 10%의 정확도르 얻었을 것이므로 이 점수가 아주 나쁘지 않지만 성능을 더 높일 여지가 있음

예를 들어 입력의 스케일을 조정하면 정확도를 89.1%이상으로 높일 수 있음

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train.astype("float64"))

cross_val_score(sgd_clf, X_train_scaled, y_train, cv=3, scoring="accuracy")

---
## 3.5 오류 분석

이 절에서는 가능성이 높은 모델을 하나 찾았다고 가정하고 이 모델의 성능을 향상시킬 방법을 찾아봄

**한 가지 방법은 생성된 오류의 종류를 분석하는 것**

오차 행렬을 살펴보자면 cross_val_predict() 함수를 사용해 예측을 만들고 confusion_matrix() 함수를 호출함.

그다음 앞에서 했던 것처럼 confusion_matrix() 함수에 레이블과 예측을 전달 가능. 하지만 클래스가 2개가 아니라 10개라서 오차 행렬에 상당히 많은 숫자가 포함되므로 읽기 어려울수도..

오차 행렬을 컬러 그래프로 나타내면 분석하기가 쉬움. 그래프를 그리려면 다음과 같이 ConfusionMatrixDisplay.from_predictions() 함수를 사용

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

y_train_pred = cross_val_predict(sgd_clf, X_train_scaled, y_train, cv=3)
ConfusionMatrixDisplay.from_predictions(y_train, y_train_pred)
plt.show()

이렇게 하면 그래프가 생성.

대부분의 이미지가 주대각선에 있는데, 이 이미지가 올바르게 분류되었음을 의미한다. 5번 행과 5번 열의 대각선에 있는 셀은 다른 숫자보다 약간 더 어둡게 보임

= 이미지가 올바르게 분류됨

5번 행 5번 열의 대각선에 있는 셀은 다른 숫자보다 약간 더 어둡게 보임 = 이는 모델이 5에서 더 많은 오류를 범했거나 데이터 집합에 다른 숫자보다 5가 적기 때문

따라서 각 값을 해당 클래스(True 레이블)의 총 이미지 수로 나누어 오차 행렬을 정규화하는 것이 중요

normalize="true"로 지정하면 이 작업을 간단히 수행할 수 있음

또한 values_format=".0%" 매개변수를 지정하여 소수점 없이 백분율을 표시할 수도 있음.


In [ ]:
ConfusionMatrixDisplay.from_predictions(y_train, y_train_pred,
                                         normalize="true", values_format=".0%")
plt.show()

이제 5 이미지의 82%만이 올바륵데 분류되었다는 것을 쉽게 알 수 있다. 모델이 5 이미지에서 가장 많이 범한 오류는 8로 잘못 분류한 것인데 전체 5의 10%에서 이러한 오류가 발생했음.

하지만 8은 2%만이 5로 잘못 분류됨

= 즉, 오차 행렬은 일반적으로 대칭이 아님!!

주의 깊게 살펴보면 많은 숫자가 8로 잘못 분류되었지만 이 그래프를 보자마자 알 수 있는 것은 아님


In [ ]:
sample_weight = (y_train_pred != y_train)
ConfusionMatrixDisplay.from_predictions(y_train, y_train_pred,
                                        sample_weight=sample_weight,
                                        normalize="true", values_format=".0%")
plt.show()

이제 분류기가 어떤 종류의 오류를 범하는지 훨씬 더 명확하게 확인할 수 있음

클래스 8의 열이 매우 밝아진 것으로 보아 많은 이미지가 8로 잘못 분류되었음을 알 수 있음

사실 이는 거의 모든 클래스에서 가장 많이 발생하는 분류 오류임

하지만 이 그래프에서 백분율을 해석하는 방법에 주의해야함. 올바른 예측을 제외했다는 것을 기억해야함

예를 들어 7번 행, 9번 열의 36%는 모든 7 이미지 중 36%가 9로 잘못 분류되었다는 의미임. 실제로는 7 이미지 중 3%만이 9로 잘못 분류된 7의 56%가 실제로는 9라는 것을 알 수 있음

오차 행렬을 행 단위가 아닌 열 단위로 정규화할 수 있음

normalize="pred" 로 지정하면 예를 들어 잘못 분류된 7의 56%가 실제로는 9라는 것을 알 수 있음

각각의 오류를 분석해보면 분류기가 무슨 일을 하는지, 왜 잘못되었는지 인사이트를 얻을 수 있음. 예를 들어 오차 행렬 스타일로 3과 5의 샘플을 그려보겠음

In [ ]:
cl_a, cl_b = '3', '5'
X_aa = X_train[(y_train == cl_a) & (y_train_pred == cl_a)]
X_ab = X_train[(y_train == cl_a) & (y_train_pred == cl_b)]
X_ba = X_train[(y_train == cl_b) & (y_train_pred == cl_a)]
X_bb = X_train[(y_train == cl_b) & (y_train_pred == cl_b)]
# X_aa 어쩌고에 있는 모든 이미지를 오차 행렬 스타일로 그림

---
## 3.6 다중 레이블 분류

지금까지는 각 샘플이 하나의 클래스에만 할당되었음

하지만 분류기가 샘플마다 여러 개의 클래스를 출력해야 할 때도 있음

**다중 레이블 분류 시스템** : 여러 개의 이진 꼬리표를 출력하는 분류 시스템

간단한 예를 보자

In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

y_train_large = (y_train >= '7')
y_train_odd = (y_train.astype('int8') % 2 ==1)
y_multilabel = np.c_[y_train_large, y_train_odd]

knn_clf = KNeighborsClassifier()
knn_clf.fit(X_train, y_multilabel)

이 코드는 각 숫자 이미지에 두 개의 타깃 레이블이 담긴  y_multilabel 배열을 만듦

첫 번째는 숫자가 큰 값(7,8,9)인지 나타내고 두 번째는 홀수인지 나타냄

그 다음 코드는 KNeighborsClassifier 인스턴스를 만들고 다중 타깃 배열을 사용하여 훈련시킴

이제 예측을 만들면 레이블이 두 개 출력됨

In [ ]:
knn_clf.predict([some_digit])

올바르게 분류됨 ! → 숫자 5는 크지않고(False) 홀수(True)임

다중 레이블 분류기를 ㅍ쳥가하는 방법은 많은데 적절한지는 프로젝트에 따라 다르다

-  각 레이블의 $F_1$ 점수를 구하고 간단하게 평균 점수를 계산하는 것

In [ ]:
y_train_knn_pred = cross_val_predict(knn_clf, X_train, y_multilabel, cv=3)
f1_score(y_multilabel, y_train_knn_pred, average="macro")

실제로는 아닐 수 있지만 이 코드는 모든 레이블의 가중치가 같다고 가정한 것

특히 앨리스 사진이 밥이나 찰리 사진보다 훨씬 많다면 앨리스 사진에 대한 분류기의 점수에 더 높은 가중치를 둘 것임

간단한 방법은 클래스의 **지지도**(타깃 레이블에 속한 샘플 수)를 가중치로 주는 것

이렇게 하려면 f1_score() 함수를 호출할 때 average="weighted"로 설정하면 됨

SVC와 같이 기본적으로 다중 레이블 분류를 지원하지 않는 분류기를 사용하는 경우 한가지 가능한 전략은 레이블 당 하나의 모델을 학습시키는 것임

그러나 이 전략은 레이블 간의 의존성을 포착하기 어렵게 할 수 있음

이 문제를 해결하기 위해 모델을 체인으로 구성할 수 있음

한 모델이 예측할 때 입력 특성과 체인 앞에 있는 모델의 모든 예측을 사용함



좋은 소식은 사이킷런에 바로 이 작업을 수행하는 ClassifierChain 클래스가 있다는 것임

기본적으로 이 클래스는 훈련에 진짜 레이블을 사용하며 체인 내 위치에 따라 각 모델에 적절한 레이블을 공급

하지만 cv 하이퍼파라미터를 지정하면 교차 검증을 사용하여 훈련 세트의 모든 샘플에 대해 훈련된 각 모델에서 '깨끗한'(표본 외) 예측을 얻고, 이러한 예측을 사용해 나중에 체인 안의 모든 모델을 훈련함ㄴ

다다음 코드는 교차 검증 전략을 사용하여 ClassifierChain을 만들고 훈련하는 방법을 보여줌

In [ ]:
from sklearn.multioutput import ClassifierChain

chain_clf = ClassifierChain(SVC(), cv=3, random_state=42)
chain_clf.fit(X_train[:2000], y_multilabel[:2000])

chain_clf.predict([some_digit])

---
## 3.7 다중 출력 분류

마지막으로 알아볼 분류 작업은 다중 출력 다중 클래스 분류이다.

이는 다중 레이블 분류에서 한 레이블이 다중 클래스가 될 수 있도록 일반화한 것임 (= 값을 두 개 이상 가질 수 있음)

이를 설명하기 위해 노이즈를 제거하는 시스템을 만들어보겠음

이 시스템은 잡음이 많은 숫자 이미지르르 입력으로 받아 꺠끗한 숫자 이미지를 MNIST 이미지처럼 픽셀의 강도를 담은 배열로 출력함

분류기의 출력이 다중 레이블일고 각 레이블의 값을 여러 개 가짐

그러므로 이 예는 다중 출력 분류 시스템임

먼저 MNIST 이미지에서 추출한 훈련 세트와 테스트 세트에 넘파이의 randint() 함수를 사용하여 픽셀 강도에 잡음을 추가함. 타깃 이미지는 원본 이미지가 될 것

In [ ]:
np.random.seed(42)
noise = np.random.randint(0, 100, (len(X_train), 784))
X_train_mod = X_train + noise
noise = np.random.randint(0, 100, (len(X_test), 784))
X_test_mod = X_test + noise
y_train_mod = X_train
y_test_mod = X_test

테스트 세트에서 이미지를 하나 선택하는데 여기서 테스트 데이터를 들여다 보는 것이 잘못된 것임을 눈치채야함

왼쪽이 잡음이 섞인 입력 이미지이고 오른쪽이 깨끗한 타깃 이미지임

분류기를 훈련 시켜 이 이미지를 깨끗하게 만들어보겠음



In [ ]:
knn_clf = KNeighborsClassifier()
knn_clf.fit(X_train_mod, y_train_mod)
clean_digit = knn_clf.predict([X_test_mod[0]])
plot_digit(clean_digit)
plt.show()

타깃과 매우 비슷하다 !!!